# AIML428 - Assignment1

## Step 2 - TF Text Classifier

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from sklearn import tree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

In [95]:
df_csv = pd.read_csv('bbc_converted.csv')

feature_column = ['text']
target_column = 'category'

X_csv = df_csv[feature_column]
y_csv = df_csv[target_column]
# unique converts to an NDArray, so the sort needs to happen first.
labels = y_csv.sort_values().unique()
print(labels)

['business' 'entertainment' 'politics' 'sport' 'tech']


In [105]:
# https://xkcd.com/221/ - 4 is overused
random_seed = 221

In [106]:
def tableHeader():
    doc = "|Algorithm| label | accuracy | precision | recall | f1 |\n"
    doc += "| --- | --- | --- | --- |--- | --- |\n"
    return doc

def tableScore(labels, doc, prefix=None, y_true=None, y_prediction=None):
    rows = ""
    accuracy = accuracy_score(y_true, y_prediction)
    precisions = precision_score(y_true, y_prediction, labels=labels, average=None)
    recalls = recall_score(y_true, y_prediction, labels=labels, average=None)
    f1s = f1_score(y_true, y_prediction, labels=labels, average=None)
    for label, precision, recall, f1 in zip(labels, precisions, recalls, f1s):
        rows += f"| {prefix} | {label} | {accuracy:.4g} | {precision:.4g} | {recall:.4g} | {f1:.4g} |\n"
        prefix = " "
    return doc + rows

def tableFooter(doc):
    display(Markdown(doc))

# TermFrequency Classifiers

In [107]:
# Convert X_csv to TF.
tfidf_vectorizer=CountVectorizer()
# When converting from dataframe to scikit_learn, fit the columns, it expects an iterable.
X_tf = tfidf_vectorizer.fit_transform(X_csv.text)

# Split the data into Train and Test
X_tf_train, X_tf_test, y_tf_train, y_tf_test = train_test_split(
    X_tf, y_csv, test_size= 0.3, random_state=random_seed)

doc = tableHeader()
# Now build a decision tree
tf_tree = tree.DecisionTreeClassifier(criterion="entropy")
tf_tree = tf_tree.fit(X_tf_train, y_tf_train)

# Evaluate the tree using X_tf_train and y_tf_train
y_tf_train_prediction = tf_tree.predict(X_tf_train)
doc = tableScore(labels, doc, "TF DecisionTree Training", y_tf_train, y_tf_train_prediction)

# Now evaluate the tree using X_tf_test and y_tf_test
y_tf_prediction = tf_tree.predict(X_tf_test)
doc = tableScore(labels, doc,"TF DecisionTree Test", y_tf_test, y_tf_prediction)

# Now use Multinomial Naive Bayes - Multinomial works with text frequencies.
tf_mnb = MultinomialNB()
tf_mnb = tf_mnb.fit(X_tf_train, y_tf_train)

# Evaluate model using X_tf_train and y_tf_train
y_tf_train_prediction = tf_mnb.predict(X_tf_train)
doc = tableScore(labels, doc,"TF NaiveBayes Training", y_tf_train, y_tf_train_prediction)

# Now evaluate model using X_tf_test and y_tf_test
y_tf_prediction = tf_mnb.predict(X_tf_test)
doc = tableScore(labels, doc,"TF NaiveBayes Test", y_tf_test, y_tf_prediction)

# Now use Support Vector Machines.
tf_svm = SVC(random_state=random_seed)
tf_svm = tf_svm.fit(X_tf_train, y_tf_train)

# Evaluate model using X_tf_train and y_tf_train
y_tf_train_prediction = tf_svm.predict(X_tf_train)
doc = tableScore(labels, doc,"TF SVM Training", y_tf_train, y_tf_train_prediction)

# Now evaluate model using X_tf_test and y_tf_test
y_tf_prediction = tf_svm.predict(X_tf_test)
doc = tableScore(labels, doc,"TF SVM Test", y_tf_test, y_tf_prediction)

tableFooter(doc)


|Algorithm| label | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- |
| TF DecisionTree Training | business | 1 | 1 | 1 | 1 |
|   | entertainment | 1 | 1 | 1 | 1 |
|   | politics | 1 | 1 | 1 | 1 |
|   | sport | 1 | 1 | 1 | 1 |
|   | tech | 1 | 1 | 1 | 1 |
| TF DecisionTree Test | business | 0.7784 | 0.6937 | 0.7303 | 0.7115 |
|   | entertainment | 0.7784 | 0.7381 | 0.7686 | 0.753 |
|   | politics | 0.7784 | 0.7194 | 0.8 | 0.7576 |
|   | sport | 0.7784 | 0.915 | 0.8805 | 0.8974 |
|   | tech | 0.7784 | 0.8444 | 0.6847 | 0.7562 |
| TF NaiveBayes Training | business | 0.9929 | 0.9972 | 0.9888 | 0.993 |
|   | entertainment | 0.9929 | 0.9925 | 0.9925 | 0.9925 |
|   | politics | 0.9929 | 0.9932 | 0.9932 | 0.9932 |
|   | sport | 0.9929 | 1 | 0.9972 | 0.9986 |
|   | tech | 0.9929 | 0.9796 | 0.9931 | 0.9863 |
| TF NaiveBayes Test | business | 0.9671 | 0.9728 | 0.9408 | 0.9565 |
|   | entertainment | 0.9671 | 1 | 0.9174 | 0.9569 |
|   | politics | 0.9671 | 0.9531 | 0.976 | 0.9644 |
|   | sport | 0.9671 | 1 | 1 | 1 |
|   | tech | 0.9671 | 0.9024 | 1 | 0.9487 |
| TF SVM Training | business | 0.9801 | 0.9699 | 0.9916 | 0.9807 |
|   | entertainment | 0.9801 | 0.9736 | 0.9736 | 0.9736 |
|   | politics | 0.9801 | 0.9858 | 0.9486 | 0.9668 |
|   | sport | 0.9801 | 0.9778 | 1 | 0.9888 |
|   | tech | 0.9801 | 0.9965 | 0.9793 | 0.9878 |
| TF SVM Test | business | 0.9311 | 0.8876 | 0.9868 | 0.9346 |
|   | entertainment | 0.9311 | 0.9722 | 0.8678 | 0.917 |
|   | politics | 0.9311 | 0.9386 | 0.856 | 0.8954 |
|   | sport | 0.9311 | 0.9568 | 0.9748 | 0.9657 |
|   | tech | 0.9311 | 0.913 | 0.9459 | 0.9292 |


# TermFrequency-InverseDocumentFrequency Classifiers

In [108]:
# Convert X_csv to TF.
tfidf_vectorizer=TfidfVectorizer()
# When converting from dataframe to scikit_learn, fit the columns, it expects an iterable.
X_tfidf = tfidf_vectorizer.fit_transform(X_csv.text)

# Split the data into Train and Test
X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test = train_test_split(
    X_tfidf, y_csv, test_size= 0.3, random_state=random_seed)

doc = tableHeader()

# Now build a decision tree
tfidf_tree = tree.DecisionTreeClassifier(criterion="entropy")
tfidf_tree = tfidf_tree.fit(X_tfidf_train, y_tfidf_train)

# Evaluate the tree using X_tfidf_train and y_tfidf_train
y_tfidf_train_prediction = tfidf_tree.predict(X_tfidf_train)
train_accuracy = accuracy_score(y_tfidf_train, y_tfidf_train_prediction)
doc = tableScore(
    labels, doc, "TFIDF DecisionTree Training", y_tfidf_train, y_tfidf_train_prediction)

# Now evaluate the tree using X_tfidf_test and y_tfidf_test
y_tfidf_prediction = tfidf_tree.predict(X_tfidf_test)
doc = tableScore(labels, doc, "TFIDF DecisionTree Test", y_tfidf_test, y_tfidf_prediction)

# Now use Multinomial Naive Bayes - Multinomial works with text frequencies.
tfidf_mnb = MultinomialNB()
tfidf_mnb = tfidf_mnb.fit(X_tfidf_train, y_tfidf_train)

# Evaluate model using X_tfidf_train and y_tfidf_train
y_tfidf_train_prediction = tfidf_mnb.predict(X_tfidf_train)
doc = tableScore(
    labels, doc, "TFIDF NaiveBayes Training", y_tfidf_train, y_tfidf_train_prediction)

# Now evaluate model using X_tfidf_test and y_tfidf_test
y_tfidf_prediction = tfidf_mnb.predict(X_tfidf_test)
doc = tableScore(
    labels, doc, "TFIDF NaiveBayes Test", y_tfidf_test, y_tfidf_prediction)

# Now use Support Vector Machines.
tfidf_svm = SVC(random_state=random_seed)
tfidf_svm = tfidf_svm.fit(X_tfidf_train, y_tfidf_train)

# Evaluate model using X_tfidf_train and y_tfidf_train
y_tfidf_train_prediction = tfidf_svm.predict(X_tfidf_train)
doc = tableScore(
    labels, doc, "TFIDF SVM Training", y_tfidf_train, y_tfidf_train_prediction)

# Now evaluate model using X_tfidf_test and y_tfidf_test
y_tfidf_prediction = tfidf_svm.predict(X_tfidf_test)
doc = tableScore(
    labels, doc, "TFIDF SVM Test", y_tfidf_test, y_tfidf_prediction)

tableFooter(doc)


|Algorithm| label | accuracy | precision | recall | f1 |
| --- | --- | --- | --- |--- | --- |
| TFIDF DecisionTree Training | business | 1 | 1 | 1 | 1 |
|   | entertainment | 1 | 1 | 1 | 1 |
|   | politics | 1 | 1 | 1 | 1 |
|   | sport | 1 | 1 | 1 | 1 |
|   | tech | 1 | 1 | 1 | 1 |
| TFIDF DecisionTree Test | business | 0.759 | 0.7123 | 0.6842 | 0.698 |
|   | entertainment | 0.759 | 0.7302 | 0.7603 | 0.7449 |
|   | politics | 0.759 | 0.7328 | 0.768 | 0.75 |
|   | sport | 0.759 | 0.8792 | 0.8239 | 0.8506 |
|   | tech | 0.759 | 0.7241 | 0.7568 | 0.7401 |
| TFIDF NaiveBayes Training | business | 0.982 | 0.9861 | 0.9916 | 0.9889 |
|   | entertainment | 0.982 | 0.996 | 0.9283 | 0.9609 |
|   | politics | 0.982 | 0.9698 | 0.9897 | 0.9797 |
|   | sport | 0.982 | 0.9804 | 0.9972 | 0.9887 |
|   | tech | 0.982 | 0.9796 | 0.9931 | 0.9863 |
| TFIDF NaiveBayes Test | business | 0.9371 | 0.9363 | 0.9671 | 0.9515 |
|   | entertainment | 0.9371 | 1 | 0.7521 | 0.8585 |
|   | politics | 0.9371 | 0.9231 | 0.96 | 0.9412 |
|   | sport | 0.9371 | 0.9464 | 1 | 0.9725 |
|   | tech | 0.9371 | 0.8934 | 0.982 | 0.9356 |
| TFIDF SVM Training | business | 0.9994 | 1 | 0.9972 | 0.9986 |
|   | entertainment | 0.9994 | 1 | 1 | 1 |
|   | politics | 0.9994 | 1 | 1 | 1 |
|   | sport | 0.9994 | 1 | 1 | 1 |
|   | tech | 0.9994 | 0.9966 | 1 | 0.9983 |
| TFIDF SVM Test | business | 0.9686 | 0.9202 | 0.9868 | 0.9524 |
|   | entertainment | 0.9686 | 0.9914 | 0.9504 | 0.9705 |
|   | politics | 0.9686 | 0.9833 | 0.944 | 0.9633 |
|   | sport | 0.9686 | 0.9876 | 1 | 0.9938 |
|   | tech | 0.9686 | 0.9722 | 0.9459 | 0.9589 |


Why does TF/IDF produce worse results? It does generally produce worse results. In DecisionTree and NaiveBayes. SVM tends to swing the other way.